In [1]:
import re, json
import curl_cffi.requests as requests
from parsel import Selector

session = requests.Session(impersonate="chrome124")
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                         "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
           "Accept-Language": "en-US,en;q=0.9"}
IMDB_ID = "tt0111161"  # The Shawshank Redemption

In [2]:
# 1. Resolve the film page; the final URL carries the slug.
r = session.get(f"https://letterboxd.com/imdb/{IMDB_ID}/", headers=HEADERS, timeout=15)
r.raise_for_status()
m = re.search(r"/film/([^/]+)/", r.url)
assert m, f"no film page for {IMDB_ID} (is it a TV series? Letterboxd is film-only) -- resolved {r.url}"
slug = m.group(1)
print("resolved URL:", r.url, "\nslug:", slug)

resolved URL: https://letterboxd.com/film/the-shawshank-redemption/ 
slug: the-shawshank-redemption


In [3]:
raw = Selector(text=r.text).css('script[type="application/ld+json"]::text').get()
ld = json.loads(raw.replace("/* <![CDATA[ */", "").replace("/* ]]> */", "").strip())
print("JSON-LD keys:", list(ld.keys()))
print(json.dumps(ld.get("aggregateRating"), indent=2))     # <- the bit we want

JSON-LD keys: ['image', '@type', 'director', 'description', 'inLanguage', 'productionCompany', '@context', 'url', 'duration', 'actor', 'dateCreated', 'name', 'genre', '@id', 'countryOfOrigin', 'aggregateRating']
{
  "bestRating": 5,
  "reviewCount": 309483,
  "@type": "AggregateRating",
  "ratingValue": 4.59,
  "description": "The Letterboxd rating is a weighted average score for a movie based on all ratings cast to date by our members.",
  "ratingCount": 2905115,
  "worstRating": 0.5
}


In [4]:
agg = ld.get("aggregateRating") or {}
print("name  :", ld.get("name"))
print("rating:", agg.get("ratingValue"), "from", agg.get("ratingCount"), "ratings")

name  : The Shawshank Redemption
rating: 4.59 from 2905115 ratings


In [5]:
csi = {**HEADERS, "Accept": "*/*", "Referer": r.url}   # csi endpoints want a Referer
h = session.get(f"https://letterboxd.com/csi/film/{slug}/rating-histogram/", headers=csi, timeout=15)
print("fragment preview:\n", h.text[:300], "...\n")

for col in Selector(text=h.text).css("tr.column"):
    star  = col.css("th._sr-only::text").get(default="").strip()
    title = col.css("a.barcolumn::attr(title)").get(default="")   # e.g. "12,345 ... ratings"
    m = re.search(r"([\d,]+)", title)
    print(f"{star:>6}: {int(m.group(1).replace(',', '')) if m else 0:,}")

fragment preview:
 
<div class="rating-histogram"> <div class="layout"> <svg xmlns="http://www.w3.org/2000/svg" role="graphics-symbol" class="glyph stars -start -rating" width="9" height="9" viewBox="0 0 9 9" aria-label="★"> <title>★</title><path transform="translate(0, 0)" fill-rule="evenodd" d="M5.065.45c-.22-.61-.9 ...

half-★: 1,838
     ★: 3,495
    ★½: 2,223
    ★★: 12,373
   ★★½: 13,608
   ★★★: 95,303
  ★★★½: 133,074
  ★★★★: 581,610
 ★★★★½: 532,826
 ★★★★★: 1,528,771


In [6]:
s = session.get(f"https://letterboxd.com/csi/film/{slug}/stats/", headers=csi, timeout=15)
ss = Selector(text=s.text)

def stat(css_class):
    label = ss.css(f"div.{css_class}::attr(aria-label)").get() or ""   # "Watched by 1,234,567 members"
    m = re.search(r"([\d,]+)", label)
    return int(m.group(1).replace(",", "")) if m else None

for cls, name in [("-watches", "watches"), ("-lists", "lists"), ("-likes", "likes")]:
    print(f"{name:>8}:", stat(cls))

 watches: 4120461
   lists: 479879
   likes: 1648301
